# 02 · KV cache management with Mooncake

The KV cache holds the attention state of every token a request has seen. It lives in GPU memory, and on this RTX 5090 there's room for **209,472 tokens**. When prompts don't fit, vLLM evicts old cache blocks, and anything evicted has to be **recomputed** (prefilled again) if the same prompt comes back.

[Mooncake](https://github.com/kvcache-ai/Mooncake) adds a second tier: a KV store in CPU memory. vLLM's `MooncakeStoreConnector` writes KV blocks there as they're computed, and on a GPU cache miss it can **load** them back instead of recomputing.

This notebook measures what that buys on one GPU, and what it costs. All numbers are read from `results/02_kv_cache_mooncake/`.

## How much KV cache is a token?

For Qwen2.5 7B: 28 layers × 4 KV heads × 128 dims × 2 (key and value) × 2 bytes (BF16) = **57,344 bytes per token** (56 KiB).

That matches vLLM's own numbers: 11.19 GiB of KV cache memory ÷ 209,472 tokens = 57,350 bytes per token.

| Tier | Size | Tokens of KV |
|---|---|---|
| GPU (vLLM's KV cache) | 11.19 GiB | 209,472 |
| Mooncake store (CPU memory, `vllm/mooncake/mooncake_config.json`) | 32 GB | ~558,000 |

## The experiment

`scripts/kv_cache_experiment.py` sends **48 distinct documents of ~5,950 tokens** (~286k tokens in total, more than the GPU cache) one at a time, with 8 output tokens each so the measurement is all about the prompt:

| Round | What | Expected without Mooncake | Expected with Mooncake |
|---|---|---|---|
| **fill** | every document once | all computed | all computed (and written to the store) |
| **revisit** | the same 48 again, same order | evicted from GPU → recomputed | loaded from CPU memory |
| **hot** | the last 5 again | still on the GPU → GPU cache hit | still on the GPU → GPU cache hit |

The same script ran on the same machine twice: once with the Phase 1 engine config (`baseline`), once with `vllm/configs/mooncake_store.yaml`, which is the baseline plus one change: the Mooncake connector.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "loadtest"))
from results_logger import load_runs, summarize

runs = load_runs("02_kv_cache_mooncake")
experiments = {r["config"]["label"]: r for r in runs if r["name"].startswith("kv_experiment_")}
traces = {r["config"]["label"]: r for r in runs if r["name"].startswith("trace_single_requests")}
print(f"experiments: {sorted(experiments)} | traces: {sorted(traces)}")
if runs:
    env = runs[-1]["environment"]
    gpu = env["gpus"][0] if env["gpus"] else {"name": "no GPU", "memory_mib": 0}
    print(f"hardware: {gpu['name']} ({gpu['memory_mib'] / 1024:.0f} GB) | model: {runs[-1]['config'].get('model')} | vLLM: {runs[-1]['config'].get('vllm_image')}")

## Results by round

In [ ]:
if not experiments:
    print("No experiment runs yet: run scripts/run_phase2.sh on the GPU box, then copy results/ back.")
else:
    print(f"{'':<10}{'round':<9}{'TTFT p50':>10}{'TTFT p95':>10}{'prefill p50':>13}{'from GPU':>10}{'from Mooncake':>15}{'computed':>10}")
    for label in ("baseline", "mooncake"):
        if label not in experiments:
            continue
        for rnd, s in experiments[label]["metrics"]["summary"].items():
            computed = 1 - s["gpu_hit_ratio"] - s["mooncake_hit_ratio"]
            print(f"{label:<10}{rnd:<9}{s['ttft_s']['p50'] * 1000:>7.0f} ms{s['ttft_s']['p95'] * 1000:>7.0f} ms"
                  f"{s['prefill_s']['p50'] * 1000:>10.0f} ms{s['gpu_hit_ratio']:>10.0%}{s['mooncake_hit_ratio']:>15.0%}{computed:>10.0%}")
        print()

In [ ]:
if experiments:
    fig, ax = plt.subplots(figsize=(11, 4))
    for label, color in (("baseline", "#d94801"), ("mooncake", "#2171b5")):
        if label in experiments:
            rows = experiments[label]["metrics"]["requests"]
            ax.plot(range(1, len(rows) + 1), [r["client"]["ttft_s"] * 1000 for r in rows], ".-", color=color, label=label, lw=1, ms=4)
    n = experiments[next(iter(experiments))]["load"]["docs"]
    for x, name in ((0, "fill"), (n, "revisit"), (2 * n, "hot")):
        ax.axvline(x + 0.5, color="#999", lw=0.8, ls="--")
        ax.text(x + 1.5, ax.get_ylim()[1] * 0.95, name, va="top", color="#555")
    ax.set_xlabel("request (in order sent)")
    ax.set_ylabel("time to first token (ms)")
    ax.set_title("TTFT for every request, with and without Mooncake")
    ax.legend()
    plt.tight_layout()
    plt.show()

## What Mooncake moved

`save_put` writes KV into the store after it's computed; `load_get` reads it back into the GPU on a cache miss. Bandwidth = bytes ÷ time spent in the operation.

In [ ]:
if "mooncake" in experiments:
    print(f"{'round':<9}{'operation':<13}{'calls':>7}{'GiB':>9}{'time':>10}{'GiB/s':>8}")
    for rnd, s in experiments["mooncake"]["metrics"]["summary"].items():
        for op, m in s["mooncake"].items():
            if m["calls"]:
                gib = m["bytes"] / 2**30
                rate = gib / m["time_s"] if m["time_s"] else float("nan")
                print(f"{rnd:<9}{op:<13}{m['calls']:>7.0f}{gib:>9.2f}{m['time_s']:>8.2f} s{rate:>8.1f}")
else:
    print("No Mooncake run yet.")

## Does Mooncake cost anything when the cache is big enough?

The Phase 1 single-request trace, rerun with and without Mooncake on the same machine. Here every prompt fits in the GPU cache, so Mooncake can only add overhead (writing every new block to the store).

In [ ]:
if traces:
    labels = [l for l in ("baseline", "mooncake") if l in traces]
    prompt_ids = list(traces[labels[0]]["metrics"]["summary"])
    print(f"{'':<14}" + "".join(f"{l + ' TTFT':>16}{l + ' e2e':>16}" for l in labels))
    for p in prompt_ids:
        row = "".join(f"{traces[l]['metrics']['summary'][p]['ttft_s']['p50'] * 1000:>13.1f} ms"
                      f"{traces[l]['metrics']['summary'][p]['e2e_s']['p50'] * 1000:>13.0f} ms" for l in labels)
        print(f"{p:<14}{row}")
else:
    print("No trace runs yet.")

## Grafana

*(added after the GPU session)*

## What happened

*(written after the GPU session, from the numbers above)*

## Phase 2 exit criteria

- [x] Mooncake store wired into vLLM through `MooncakeStoreConnector`, as one config change (`vllm/configs/mooncake_store.yaml`)
- [x] Experiment with a working set bigger than the GPU KV cache, measured per request
- [x] Dashboard row for KV cache tiers and Mooncake transfers
- [ ] Phase 2 session on the RTX 5090: baseline and Mooncake runs on the same machine
- [ ] With and without comparison explained: what Mooncake buys and what it costs